In [ ]:
print('Hello')

In [ ]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
from pathlib import Path

DATASET = Path("/kaggle/input/datasets/smnahian/lowlight")

print("Dataset exists:", DATASET.exists())
print("Contents:")

for item in DATASET.iterdir():
    print(item)

In [ ]:
from pathlib import Path
import pandas as pd

DATASET = Path("/kaggle/input/datasets/smnahian/lowlight")

IMAGE_DIR = DATASET / "Images"
ANNOTATION_DIR = DATASET / "Annotaions"
SPLIT_FILE = ANNOTATION_DIR / "imageclasslist.txt"

print("Image directory exists:", IMAGE_DIR.exists())
print("Annotation directory exists:", ANNOTATION_DIR.exists())
print("Split file exists:", SPLIT_FILE.exists())

print("\nImage folders:")
for folder in sorted(IMAGE_DIR.iterdir()):
    print(folder.name)

print("\nAnnotation folders:")
for folder in sorted(ANNOTATION_DIR.iterdir()):
    print(folder.name)

In [ ]:
from pathlib import Path
import pandas as pd

DATASET = Path("/kaggle/input/datasets/smnahian/lowlight")
SPLIT_FILE = DATASET / "Annotaions" / "imageclasslist.txt"

# Read the file:
# Data rows are whitespace-separated.
df = pd.read_csv(
    SPLIT_FILE,
    sep=r"\s+",
    skiprows=1,
    header=None,
    names=["Name", "Class", "Light", "In_Out", "Split"]
)

# Convert numeric columns
df["Class"] = pd.to_numeric(df["Class"], errors="coerce")
df["Light"] = pd.to_numeric(df["Light"], errors="coerce")
df["In_Out"] = pd.to_numeric(df["In_Out"], errors="coerce")
df["Split"] = pd.to_numeric(df["Split"], errors="coerce")

# Remove any invalid rows
df = df.dropna(subset=["Name", "Class", "Split"])

# Convert to integers
df["Class"] = df["Class"].astype(int)
df["Light"] = df["Light"].astype(int)
df["In_Out"] = df["In_Out"].astype(int)
df["Split"] = df["Split"].astype(int)

print("Columns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
print(df.head())

print("\nTotal rows:", len(df))

In [ ]:
# ExDark class IDs
vehicle_classes = {
    1: "Bicycle",
    2: "Boat",
    4: "Bus",
    5: "Car",
    10: "Motorbike"
}

# Keep only vehicle classes
vehicle_df = df[df["Class"].isin(vehicle_classes.keys())].copy()

# Convert class IDs to names
vehicle_df["Class_Name"] = vehicle_df["Class"].map(vehicle_classes)

# Convert split IDs
split_names = {
    1: "Train",
    2: "Validation",
    3: "Test"
}

vehicle_df["Split_Name"] = vehicle_df["Split"].map(split_names)

print("Total vehicle images:", len(vehicle_df))

print("\nClass counts:")
print(vehicle_df["Class_Name"].value_counts().sort_index())

print("\nSplit counts:")
print(vehicle_df["Split_Name"].value_counts())

print("\nClass × Split:")
print(
    pd.crosstab(
        vehicle_df["Class_Name"],
        vehicle_df["Split_Name"]
    )
)

In [ ]:
from pathlib import Path

image_dir = DATASET / "Images" / "Boat"
annotation_dir = DATASET / "Annotaions" / "Boat"

# Get first image
images = list(image_dir.glob("*"))
image_file = images[0]

print("Image:")
print(image_file)

# Annotation filename = complete image filename + ".txt"
annotation_file = annotation_dir / (image_file.name + ".txt")

print("\nAnnotation:")
print(annotation_file)

print("\nAnnotation exists:", annotation_file.exists())

if annotation_file.exists():
    print("\nAnnotation content:\n")
    print(annotation_file.read_text(errors="ignore"))

In [ ]:
from pathlib import Path
from collections import Counter

# =========================
# Dataset paths
# =========================

DATASET = Path("/kaggle/input/datasets/smnahian/lowlight")

IMAGE_DIR = DATASET / "Images"
ANNOTATION_DIR = DATASET / "Annotaions"

# Only these 5 classes
VEHICLE_CLASSES = {
    "Bicycle",
    "Boat",
    "Bus",
    "Car",
    "Motorbike"
}

# =========================
# Counters
# =========================

total_images = 0
annotation_found = 0
annotation_missing = 0

images_with_vehicle = 0
images_with_non_vehicle = 0

vehicle_objects = Counter()
non_vehicle_objects = Counter()

missing_annotations = []
empty_annotations = []
invalid_annotations = []

# =========================
# Scan vehicle image folders
# =========================

for class_name in sorted(VEHICLE_CLASSES):

    image_folder = IMAGE_DIR / class_name
    annotation_folder = ANNOTATION_DIR / class_name

    print(f"Scanning {class_name}...")

    for image_file in image_folder.iterdir():

        if not image_file.is_file():
            continue

        total_images += 1

        # Annotation name:
        # 2015_01231.jpg -> 2015_01231.jpg.txt
        annotation_file = annotation_folder / (image_file.name + ".txt")

        if not annotation_file.exists():
            annotation_missing += 1
            missing_annotations.append(str(image_file))
            continue

        annotation_found += 1

        # Read annotation
        lines = annotation_file.read_text(
            errors="ignore"
        ).splitlines()

        objects_in_image = []
        has_vehicle = False
        has_non_vehicle = False

        for line in lines:

            line = line.strip()

            # Skip metadata line
            if not line or line.startswith("%"):
                continue

            parts = line.split()

            # Expected:
            # Class l t w h + 7 values
            if len(parts) < 5:
                invalid_annotations.append(
                    (str(annotation_file), line)
                )
                continue

            object_class = parts[0]

            objects_in_image.append(object_class)

            if object_class in VEHICLE_CLASSES:
                vehicle_objects[object_class] += 1
                has_vehicle = True
            else:
                non_vehicle_objects[object_class] += 1
                has_non_vehicle = True

        if not objects_in_image:
            empty_annotations.append(str(annotation_file))

        if has_vehicle:
            images_with_vehicle += 1

        if has_non_vehicle:
            images_with_non_vehicle += 1


# =========================
# Results
# =========================

print("\n" + "=" * 50)
print("DATASET VALIDATION RESULT")
print("=" * 50)

print(f"\nTotal vehicle-class images: {total_images}")

print(f"Annotations found:          {annotation_found}")
print(f"Annotations missing:        {annotation_missing}")

print(f"\nImages containing vehicle:  {images_with_vehicle}")
print(f"Images containing non-vehicle: {images_with_non_vehicle}")

print("\nVehicle object counts:")
for name, count in vehicle_objects.items():
    print(f"  {name}: {count}")

print("\nNon-vehicle object counts:")
for name, count in non_vehicle_objects.most_common():
    print(f"  {name}: {count}")

print("\nEmpty annotations:", len(empty_annotations))
print("Invalid annotations:", len(invalid_annotations))

# Show missing annotation examples
if missing_annotations:
    print("\nFirst 10 missing annotation examples:")
    for x in missing_annotations[:10]:
        print(" ", x)

# Show invalid annotation examples
if invalid_annotations:
    print("\nFirst 10 invalid annotation examples:")
    for x in invalid_annotations[:10]:
        print(" ", x)

In [ ]:
# Check images with missing/empty annotations

print("Missing annotation images:")
for x in missing_annotations:
    print(x)

print("\nEmpty annotation files:")
for x in empty_annotations:
    print(x)

In [ ]:
from pathlib import Path

# =========================
# Paths
# =========================

DATASET = Path("/kaggle/input/datasets/smnahian/lowlight")

IMAGE_DIR = DATASET / "Images"
ANNOTATION_DIR = DATASET / "Annotaions"

# Output directory
OUTPUT_DIR = Path("/kaggle/working/filtered_annotations")

# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# =========================
# Vehicle classes
# =========================

VEHICLE_CLASSES = {
    "Bicycle",
    "Boat",
    "Bus",
    "Car",
    "Motorbike"
}


# =========================
# Counters
# =========================

total_annotations = 0
filtered_annotations = 0
removed_objects = 0
images_with_vehicle = 0
empty_after_filtering = 0


# =========================
# Process annotations
# =========================

for class_folder in sorted(VEHICLE_CLASSES):

    input_folder = ANNOTATION_DIR / class_folder

    # Create corresponding output folder
    output_folder = OUTPUT_DIR / class_folder
    output_folder.mkdir(parents=True, exist_ok=True)

    print(f"Processing {class_folder}...")

    # Find all annotation files
    for annotation_file in input_folder.glob("*.txt"):

        # Read annotation
        lines = annotation_file.read_text(
            errors="ignore"
        ).splitlines()

        total_annotations += 1

        filtered_lines = []

        for line in lines:

            line = line.strip()

            # Keep metadata line
            if line.startswith("%"):
                filtered_lines.append(line)
                continue

            # Skip blank lines
            if not line:
                continue

            parts = line.split()

            # Need at least class + l + t + w + h
            if len(parts) < 5:
                continue

            object_class = parts[0]

            # Keep only vehicle objects
            if object_class in VEHICLE_CLASSES:

                filtered_lines.append(line)
                filtered_annotations += 1

            else:

                removed_objects += 1

        # Save filtered annotation
        output_file = output_folder / annotation_file.name

        if len(filtered_lines) > 1:
            images_with_vehicle += 1

        else:
            empty_after_filtering += 1

        output_file.write_text(
            "\n".join(filtered_lines)
        )


# =========================
# Results
# =========================

print("\n" + "=" * 50)
print("ANNOTATION FILTERING COMPLETE")
print("=" * 50)

print(f"Original annotation files processed: {total_annotations}")
print(f"Vehicle annotations kept:             {filtered_annotations}")
print(f"Non-vehicle annotations removed:      {removed_objects}")
print(f"Files containing vehicle objects:     {images_with_vehicle}")
print(f"Files empty after filtering:          {empty_after_filtering}")

print("\nFiltered annotations saved to:")
print(OUTPUT_DIR)

In [ ]:
# Check the same Boat annotation we inspected earlier

original_file = (
    ANNOTATION_DIR
    / "Boat"
    / "2015_00658.jpg.txt"
)

filtered_file = (
    OUTPUT_DIR
    / "Boat"
    / "2015_00658.jpg.txt"
)

print("Original annotation:\n")
print(original_file.read_text(errors="ignore"))

print("\n" + "=" * 50)

print("Filtered annotation:\n")
print(filtered_file.read_text(errors="ignore"))

In [ ]:
from pathlib import Path
import shutil

# =========================
# Paths
# =========================

DATASET = Path("/kaggle/input/datasets/smnahian/lowlight")

IMAGE_DIR = DATASET / "Images"
FILTERED_ANNOTATION_DIR = Path("/kaggle/working/filtered_annotations")

# New clean dataset
CLEAN_DATASET = Path("/kaggle/working/vehicle_dataset")

CLEAN_IMAGE_DIR = CLEAN_DATASET / "Images"
CLEAN_ANNOTATION_DIR = CLEAN_DATASET / "Annotations"

# Vehicle classes
VEHICLE_CLASSES = {
    "Bicycle",
    "Boat",
    "Bus",
    "Car",
    "Motorbike"
}

# Create output folders
for class_name in VEHICLE_CLASSES:
    (CLEAN_IMAGE_DIR / class_name).mkdir(parents=True, exist_ok=True)
    (CLEAN_ANNOTATION_DIR / class_name).mkdir(parents=True, exist_ok=True)


# =========================
# Copy valid image + annotation pairs
# =========================

copied_images = 0
copied_annotations = 0
skipped_images = 0

for class_name in sorted(VEHICLE_CLASSES):

    image_folder = IMAGE_DIR / class_name
    annotation_folder = FILTERED_ANNOTATION_DIR / class_name

    output_image_folder = CLEAN_IMAGE_DIR / class_name
    output_annotation_folder = CLEAN_ANNOTATION_DIR / class_name

    for image_file in image_folder.iterdir():

        if not image_file.is_file():
            continue

        # Filtered annotation uses:
        # image.jpg -> image.jpg.txt
        annotation_file = annotation_folder / (image_file.name + ".txt")

        # If annotation doesn't exist, skip
        if not annotation_file.exists():
            skipped_images += 1
            continue

        # Read filtered annotation
        annotation_text = annotation_file.read_text(errors="ignore").strip()

        # Remove metadata line and check whether
        # at least one actual vehicle object exists
        object_lines = [
            line.strip()
            for line in annotation_text.splitlines()
            if line.strip() and not line.strip().startswith("%")
        ]

        # If no vehicle object, skip
        if len(object_lines) == 0:
            skipped_images += 1
            continue

        # Copy image
        shutil.copy2(
            image_file,
            output_image_folder / image_file.name
        )

        # Copy filtered annotation
        shutil.copy2(
            annotation_file,
            output_annotation_folder / annotation_file.name
        )

        copied_images += 1
        copied_annotations += 1


# =========================
# Results
# =========================

print("=" * 55)
print("CLEAN VEHICLE DATASET CREATED")
print("=" * 55)

print(f"Images copied:       {copied_images}")
print(f"Annotations copied:  {copied_annotations}")
print(f"Images skipped:      {skipped_images}")

print("\nDataset location:")
print(CLEAN_DATASET)

In [ ]:
print("\nClass-wise image counts:")

for class_name in sorted(VEHICLE_CLASSES):
    folder = CLEAN_IMAGE_DIR / class_name
    count = len([x for x in folder.iterdir() if x.is_file()])
    print(f"{class_name}: {count}")

print("\nClass-wise annotation counts:")

for class_name in sorted(VEHICLE_CLASSES):
    folder = CLEAN_ANNOTATION_DIR / class_name
    count = len([x for x in folder.iterdir() if x.is_file()])
    print(f"{class_name}: {count}")

In [ ]:
!git clone https://github.com/caiyuanhao1998/Retinexformer.git
%cd Retinexformer
!ls

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
!cat setup.py

In [ ]:
!find Enhancement -maxdepth 2 -type f | head -30

In [ ]:
!sed -n '1,240p' Enhancement/test_from_dataset.py

In [ ]:
!sed -n '1,240p' Enhancement/utils.py

In [ ]:
!grep -n -i -A5 -B5 "pretrained" README.md | head -80

!find . -maxdepth 3 -type f | grep -E "\.(pth|pt|ckpt)$"

In [ ]:
!mkdir -p pretrained_weights

In [ ]:
!pip install -q gdown

In [ ]:
!ls -lh pretrained_weights/

In [ ]:
import requests

url = "https://drive.google.com/drive/folders/1ynK5hfQachzc8y96ZumhkPPDXzHJwaQV"

r = requests.get(url)

print("Status:", r.status_code)
print("Page length:", len(r.text))
print("LOL_v2_real.pth found:", "LOL_v2_real.pth" in r.text)

In [ ]:
import re
import requests

url = "https://drive.google.com/drive/folders/1ynK5hfQachzc8y96ZumhkPPDXzHJwaQV"

html = requests.get(url).text

# Find the file ID associated with LOL_v2_real.pth
pattern = r'([a-zA-Z0-9_-]{20,})[^"]{0,500}LOL_v2_real\.pth'
matches = re.findall(pattern, html)

print("Possible IDs found:")
for x in matches[:10]:
    print(x)

In [ ]:
!gdown "https://drive.google.com/uc?id=1xDwQtTCj3tlAVCTJgYrzonBGVwqeOhKu" \
    -O pretrained_weights/LOL_v2_real.pth

In [ ]:
import os

weight_path = "pretrained_weights/LOL_v2_real.pth"

print("Exists:", os.path.exists(weight_path))

if os.path.exists(weight_path):
    print("Size:", round(os.path.getsize(weight_path) / (1024**2), 2), "MB")

In [ ]:
!pip install -q lmdb
import sys
import torch

sys.path.insert(0, "/kaggle/working/Retinexformer")

from basicsr.models import create_model
from basicsr.utils.options import parse

print("RetinexFormer imports: OK")
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

In [ ]:
import yaml

opt_path = "Options/RetinexFormer_LOL_v2_real.yml"

with open(opt_path, "r") as f:
    config = yaml.safe_load(f)

print("Configuration loaded successfully")
print("Network:", config["network_g"].get("type"))

In [ ]:
from pathlib import Path

test_image = Path(
    "/kaggle/working/vehicle_dataset/Images/Boat/2015_01231.jpg"
)

print("Image exists:", test_image.exists())
print("Image:", test_image)

In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn.functional as F

from basicsr.models import create_model
from basicsr.utils.options import parse

# ============================================================
# PATHS
# ============================================================

repo_dir = "/kaggle/working/Retinexformer"

opt_path = os.path.join(
    repo_dir,
    "Options/RetinexFormer_LOL_v2_real.yml"
)

weight_path = os.path.join(
    repo_dir,
    "pretrained_weights/LOL_v2_real.pth"
)

input_path = (
    "/kaggle/working/vehicle_dataset/Images/"
    "Boat/2015_01231.jpg"
)

output_path = (
    "/kaggle/working/test_retinexformer.jpg"
)


# ============================================================
# LOAD CONFIGURATION
# ============================================================

opt = parse(
    opt_path,
    is_train=False
)

opt["dist"] = False

model = create_model(opt).net_g


# ============================================================
# LOAD PRETRAINED WEIGHTS
# ============================================================

checkpoint = torch.load(
    weight_path,
    map_location="cpu"
)

try:

    model.load_state_dict(
        checkpoint["params"]
    )

except Exception:

    new_checkpoint = {}

    for k, v in checkpoint["params"].items():

        new_checkpoint["module." + k] = v

    model.load_state_dict(
        new_checkpoint
    )


# ============================================================
# MOVE MODEL TO GPU
# ============================================================

model = model.cuda()
model.eval()

print("=" * 60)
print("RETINEXFORMER MODEL LOADED")
print("=" * 60)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

print("Model loaded successfully.")


# ============================================================
# LOAD IMAGE
# ============================================================

img = cv2.imread(
    input_path
)

if img is None:

    raise FileNotFoundError(
        input_path
    )

print("\nOriginal image shape:", img.shape)

# BGR -> RGB
img = cv2.cvtColor(
    img,
    cv2.COLOR_BGR2RGB
)

# [0,255] -> [0,1]
img = img.astype(
    np.float32
) / 255.0

# H,W,C -> C,H,W
input_tensor = (
    torch
    .from_numpy(img)
    .permute(2, 0, 1)
)

# Add batch dimension
input_tensor = (
    input_tensor
    .unsqueeze(0)
    .cuda()
)


# ============================================================
# ORIGINAL DIMENSIONS
# ============================================================

_, _, h, w = input_tensor.shape

print(
    "Input tensor size:",
    f"{h} x {w}"
)


# ============================================================
# PADDING
# RetinexFormer requires dimensions divisible by 4
# ============================================================

factor = 4

new_h = (
    (h + factor - 1)
    // factor
) * factor

new_w = (
    (w + factor - 1)
    // factor
) * factor

pad_h = new_h - h
pad_w = new_w - w

if pad_h > 0 or pad_w > 0:

    input_tensor = F.pad(
        input_tensor,
        (0, pad_w, 0, pad_h),
        mode="reflect"
    )

print(
    "Padded tensor size:",
    f"{new_h} x {new_w}"
)


# ============================================================
# RETINEXFORMER ENHANCEMENT
# ============================================================

with torch.inference_mode():

    restored = model(
        input_tensor
    )


# ============================================================
# REMOVE PADDING
# ============================================================

restored = restored[
    :, :, :h, :w
]


# ============================================================
# CLAMP VALUES
# ============================================================

restored = torch.clamp(
    restored,
    0,
    1
)


# ============================================================
# TENSOR -> NUMPY
# ============================================================

restored = (
    restored
    .cpu()
    .squeeze(0)
    .permute(1, 2, 0)
    .numpy()
)


# ============================================================
# [0,1] -> [0,255]
# ============================================================

restored = (
    restored * 255
).astype(
    np.uint8
)


# ============================================================
# RGB -> BGR
# ============================================================

restored = cv2.cvtColor(
    restored,
    cv2.COLOR_RGB2BGR
)


# ============================================================
# SAVE IMAGE
# ============================================================

save_success = cv2.imwrite(
    output_path,
    restored
)

if not save_success:

    raise RuntimeError(
        "Failed to save enhanced image."
    )


# ============================================================
# RESULT
# ============================================================

print("\n" + "=" * 60)
print("RETINEXFORMER TEST COMPLETE")
print("=" * 60)

print("Enhancement successful.")
print("Saved to:")
print(output_path)

print(
    "Enhanced image shape:",
    restored.shape
)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

original = Image.open(
    "/kaggle/working/vehicle_dataset/Images/Boat/2015_01231.jpg"
)

enhanced = Image.open(
    "/kaggle/working/test_retinexformer.jpg"
)

print("Original size:", original.size)
print("Enhanced size:", enhanced.size)

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.imshow(original)
plt.title("Original ExDark")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(enhanced)
plt.title("RetinexFormer Enhanced")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# শুধু একটি image enhance করার function

def enhance_one(input_path, output_path):

    img = cv2.imread(input_path)

    if img is None:
        print("Image not found:", input_path)
        return

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = np.float32(img) / 255.0

    tensor = torch.from_numpy(img).permute(2, 0, 1)
    tensor = tensor.unsqueeze(0).cuda()

    _, _, h, w = tensor.shape

    # Padding
    factor = 4
    new_h = ((h + factor - 1) // factor) * factor
    new_w = ((w + factor - 1) // factor) * factor

    tensor = F.pad(
        tensor,
        (0, new_w - w, 0, new_h - h),
        mode="reflect"
    )

    # RetinexFormer
    with torch.inference_mode():
        restored = model(tensor)

    # Remove padding
    restored = restored[:, :, :h, :w]

    # Convert back
    restored = torch.clamp(restored, 0, 1)
    restored = (
        restored.cpu()
        .permute(0, 2, 3, 1)
        .squeeze(0)
        .numpy()
    )

    restored = (restored * 255).astype(np.uint8)

    # Save
    Image.fromarray(restored).save(output_path)

    print("Done!")
    print("Original:", input_path)
    print("Enhanced:", output_path)
    print("Size:", (w, h))

In [ ]:
input_path = "/kaggle/working/vehicle_dataset/Images/Boat/2015_00676.jpg"

output_path = "/kaggle/working/test_00676.jpg"

enhance_one(input_path, output_path)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

original = Image.open(input_path)
enhanced = Image.open(output_path)

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.imshow(original)
plt.title("Original ExDark")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(enhanced)
plt.title("RetinexFormer Enhanced")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm

# ============================================================
# INPUT / OUTPUT DIRECTORIES
# ============================================================

input_root = "/kaggle/working/vehicle_dataset/Images"

output_root = "/kaggle/working/enhanced_vehicle_dataset/Images"

# Create output directory
os.makedirs(output_root, exist_ok=True)

# Vehicle classes
classes = [
    "Bicycle",
    "Boat",
    "Bus",
    "Car",
    "Motorbike"
]

# ============================================================
# ENHANCEMENT FUNCTION
# ============================================================

def enhance_image(input_path, output_path):

    # Read image
    img = cv2.imread(input_path)

    if img is None:
        return False

    # BGR -> RGB
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Convert to [0, 1]
    img = np.float32(img) / 255.0

    # Convert to tensor
    input_tensor = torch.from_numpy(img).permute(2, 0, 1)
    input_tensor = input_tensor.unsqueeze(0).cuda()

    # Original dimensions
    _, _, h, w = input_tensor.shape

    # RetinexFormer requires divisible by 4
    factor = 4

    new_h = ((h + factor - 1) // factor) * factor
    new_w = ((w + factor - 1) // factor) * factor

    pad_h = new_h - h
    pad_w = new_w - w

    # Padding
    if pad_h > 0 or pad_w > 0:
        input_tensor = F.pad(
            input_tensor,
            (0, pad_w, 0, pad_h),
            mode="reflect"
        )

    # ========================================================
    # RETINEXFORMER
    # ========================================================

    with torch.inference_mode():

        restored = model(input_tensor)

    # Remove padding
    restored = restored[:, :, :h, :w]

    # Clamp
    restored = torch.clamp(restored, 0, 1)

    # Tensor -> NumPy
    restored = (
        restored
        .cpu()
        .permute(0, 2, 3, 1)
        .squeeze(0)
        .numpy()
    )

    # [0,1] -> [0,255]
    restored = (restored * 255).astype(np.uint8)

    # RGB -> BGR for OpenCV
    restored = cv2.cvtColor(restored, cv2.COLOR_RGB2BGR)

    # Save
    cv2.imwrite(output_path, restored)

    return True


# ============================================================
# PROCESS ALL IMAGES
# ============================================================

total = 0
success = 0
failed = []

for class_name in classes:

    input_dir = os.path.join(input_root, class_name)
    output_dir = os.path.join(output_root, class_name)

    os.makedirs(output_dir, exist_ok=True)

    print(f"\nProcessing {class_name}...")

    image_files = []

    for filename in os.listdir(input_dir):

        if filename.lower().endswith(
            (".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG")
        ):
            image_files.append(filename)

    image_files.sort()

    for filename in tqdm(image_files):

        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, filename)

        total += 1

        try:

            result = enhance_image(
                input_path,
                output_path
            )

            if result:
                success += 1
            else:
                failed.append(filename)

        except Exception as e:

            failed.append(
                f"{class_name}/{filename} -> {str(e)}"
            )

# ============================================================
# RESULT
# ============================================================

print("\n" + "=" * 60)
print("RETINEXFORMER ENHANCEMENT COMPLETE")
print("=" * 60)

print("Total images:", total)
print("Successfully enhanced:", success)
print("Failed:", len(failed))

print("\nOutput directory:")
print(output_root)

if failed:
    print("\nFirst 10 failed images:")
    for item in failed[:10]:
        print(item)

In [ ]:
import os

# ============================================================
# PATHS
# ============================================================

original_root = "/kaggle/working/vehicle_dataset/Images"
enhanced_root = "/kaggle/working/enhanced_vehicle_dataset/Images"

missing_log_path = "/kaggle/working/missing_enhanced_images.txt"

valid_extensions = (
    ".jpg",
    ".jpeg",
    ".png",
    ".JPG",
    ".JPEG",
    ".PNG"
)

# ============================================================
# COLLECT ORIGINAL IMAGES
# ============================================================

original_images = []

for class_name in sorted(os.listdir(original_root)):

    class_dir = os.path.join(
        original_root,
        class_name
    )

    if not os.path.isdir(class_dir):
        continue

    for filename in sorted(os.listdir(class_dir)):

        if filename.endswith(valid_extensions):

            relative_path = os.path.join(
                class_name,
                filename
            )

            original_images.append(
                relative_path
            )

# ============================================================
# FIND MISSING ENHANCED IMAGES
# ============================================================

missing_images = []

for relative_path in original_images:

    enhanced_path = os.path.join(
        enhanced_root,
        relative_path
    )

    if not os.path.exists(enhanced_path):

        missing_images.append(
            relative_path
        )

# ============================================================
# SAVE MISSING LIST
# ============================================================

with open(
    missing_log_path,
    "w"
) as f:

    for relative_path in missing_images:
        f.write(relative_path + "\n")

# ============================================================
# SUMMARY
# ============================================================

print("=" * 60)
print("MISSING ENHANCED IMAGE CHECK")
print("=" * 60)

print(
    "Total original images:",
    len(original_images)
)

print(
    "Enhanced images found:",
    len(original_images) - len(missing_images)
)

print(
    "Missing images:",
    len(missing_images)
)

print(
    "\nMissing list saved to:"
)

print(
    missing_log_path
)

print(
    "\nFirst 20 missing images:"
)

for item in missing_images[:20]:
    print(item)

In [ ]:
import os
import cv2
from collections import Counter

original_root = "/kaggle/working/vehicle_dataset/Images"
missing_log_path = "/kaggle/working/missing_enhanced_images.txt"

# ============================================================
# LOAD MISSING LIST
# ============================================================

with open(missing_log_path, "r") as f:
    missing_images = [
        line.strip()
        for line in f
        if line.strip()
    ]

# ============================================================
# INSPECT DIMENSIONS
# ============================================================

dimension_info = []
dimension_counter = Counter()

for relative_path in missing_images:

    img_path = os.path.join(
        original_root,
        relative_path
    )

    img = cv2.imread(img_path)

    if img is None:
        dimension_info.append(
            (relative_path, None, None)
        )
        continue

    h, w = img.shape[:2]

    dimension_info.append(
        (relative_path, h, w)
    )

    dimension_counter[
        (h, w)
    ] += 1

# ============================================================
# SUMMARY
# ============================================================

valid_dims = [
    (h, w)
    for _, h, w in dimension_info
    if h is not None
]

heights = [h for h, w in valid_dims]
widths = [w for h, w in valid_dims]

print("=" * 60)
print("FAILED IMAGE RESOLUTION ANALYSIS")
print("=" * 60)

print("Total missing images:", len(missing_images))
print("Readable images:", len(valid_dims))

if valid_dims:

    print(
        "\nMinimum resolution:",
        f"{min(widths)} x {min(heights)}"
    )

    print(
        "Maximum resolution:",
        f"{max(widths)} x {max(heights)}"
    )

print("\nMost common resolutions:")

for (h, w), count in dimension_counter.most_common(15):

    print(
        f"{w} x {h} -> {count} images"
    )

print("\nFirst 25 missing-image dimensions:")

for relative_path, h, w in dimension_info[:25]:

    if h is None:
        print(relative_path, "-> unreadable")
    else:
        print(
            relative_path,
            "->",
            f"{w} x {h}"
        )

In [ ]:
import os
import gc
import cv2
import numpy as np
import torch
import torch.nn.functional as F

# ============================================================
# PATHS
# ============================================================

original_root = "/kaggle/working/vehicle_dataset/Images"
enhanced_root = "/kaggle/working/enhanced_vehicle_dataset/Images"
missing_log_path = "/kaggle/working/missing_enhanced_images.txt"

retry_failed_log = "/kaggle/working/tiled_retry_failed.txt"

# ============================================================
# TILE SETTINGS
# ============================================================

TILE_SIZE = 512
OVERLAP = 32

device = next(model.parameters()).device

print("=" * 60)
print("TILED RETINEXFORMER RETRY")
print("=" * 60)

print("Device:", device)
print("Tile size:", TILE_SIZE)
print("Overlap:", OVERLAP)

# ============================================================
# LOAD MISSING LIST
# ============================================================

with open(missing_log_path, "r") as f:
    missing_images = [
        line.strip()
        for line in f
        if line.strip()
    ]

print("Images to retry:", len(missing_images))

# ============================================================
# TILE ENHANCEMENT FUNCTION
# ============================================================

def enhance_tile(tile_bgr):

    tile_rgb = cv2.cvtColor(
        tile_bgr,
        cv2.COLOR_BGR2RGB
    )

    tile_rgb = tile_rgb.astype(
        np.float32
    ) / 255.0

    tensor = (
        torch.from_numpy(tile_rgb)
        .permute(2, 0, 1)
        .unsqueeze(0)
        .to(device)
    )

    _, _, h, w = tensor.shape

    # RetinexFormer dimension compatibility
    factor = 4

    new_h = ((h + factor - 1) // factor) * factor
    new_w = ((w + factor - 1) // factor) * factor

    pad_h = new_h - h
    pad_w = new_w - w

    if pad_h > 0 or pad_w > 0:

        tensor = F.pad(
            tensor,
            (0, pad_w, 0, pad_h),
            mode="reflect"
        )

    with torch.inference_mode():

        output = model(tensor)

    output = output[
        :, :, :h, :w
    ]

    output = torch.clamp(
        output,
        0,
        1
    )

    output = (
        output
        .cpu()
        .squeeze(0)
        .permute(1, 2, 0)
        .numpy()
    )

    output = (
        output * 255.0
    ).astype(np.uint8)

    output = cv2.cvtColor(
        output,
        cv2.COLOR_RGB2BGR
    )

    del tensor

    return output


# ============================================================
# FULL IMAGE USING TILES
# ============================================================

def enhance_image_tiled(img):

    h, w = img.shape[:2]

    stride = TILE_SIZE - OVERLAP

    result = np.zeros(
        (h, w, 3),
        dtype=np.float32
    )

    weight = np.zeros(
        (h, w, 1),
        dtype=np.float32
    )

    y_positions = list(
        range(0, max(h - TILE_SIZE, 0) + 1, stride)
    )

    x_positions = list(
        range(0, max(w - TILE_SIZE, 0) + 1, stride)
    )

    if len(y_positions) == 0:
        y_positions = [0]

    if len(x_positions) == 0:
        x_positions = [0]

    if y_positions[-1] + TILE_SIZE < h:
        y_positions.append(
            max(h - TILE_SIZE, 0)
        )

    if x_positions[-1] + TILE_SIZE < w:
        x_positions.append(
            max(w - TILE_SIZE, 0)
        )

    for y in y_positions:

        for x in x_positions:

            y2 = min(
                y + TILE_SIZE,
                h
            )

            x2 = min(
                x + TILE_SIZE,
                w
            )

            tile = img[
                y:y2,
                x:x2
            ]

            enhanced_tile = enhance_tile(
                tile
            )

            th, tw = enhanced_tile.shape[:2]

            result[
                y:y+th,
                x:x+tw
            ] += enhanced_tile.astype(
                np.float32
            )

            weight[
                y:y+th,
                x:x+tw
            ] += 1.0

            del tile
            del enhanced_tile

    result /= np.maximum(
        weight,
        1e-8
    )

    result = np.clip(
        result,
        0,
        255
    ).astype(np.uint8)

    return result


# ============================================================
# RETRY MISSING IMAGES
# ============================================================

success_count = 0
failed_images = []

for idx, relative_path in enumerate(
    missing_images,
    start=1
):

    input_path = os.path.join(
        original_root,
        relative_path
    )

    output_path = os.path.join(
        enhanced_root,
        relative_path
    )

    os.makedirs(
        os.path.dirname(output_path),
        exist_ok=True
    )

    print(
        f"[{idx}/{len(missing_images)}]",
        relative_path
    )

    try:

        img = cv2.imread(
            input_path
        )

        if img is None:
            raise RuntimeError(
                "Image could not be read."
            )

        original_h, original_w = img.shape[:2]

        enhanced = enhance_image_tiled(
            img
        )

        enhanced_h, enhanced_w = enhanced.shape[:2]

        if (
            enhanced_h != original_h
            or enhanced_w != original_w
        ):

            raise RuntimeError(
                "Output dimension mismatch."
            )

        save_ok = cv2.imwrite(
            output_path,
            enhanced
        )

        if not save_ok:

            raise RuntimeError(
                "cv2.imwrite failed."
            )

        success_count += 1

        print(
            "   ✅ Success:",
            f"{original_w}x{original_h}"
        )

    except Exception as e:

        failed_images.append(
            (relative_path, str(e))
        )

        print(
            "   ❌ Failed:",
            str(e)
        )

    finally:

        if "img" in locals():
            del img

        if "enhanced" in locals():
            del enhanced

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# ============================================================
# SAVE RETRY FAILURE LOG
# ============================================================

with open(
    retry_failed_log,
    "w"
) as f:

    for relative_path, error in failed_images:

        f.write(
            relative_path
            + " -> "
            + error
            + "\n"
        )


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("TILED RETRY COMPLETE")
print("=" * 60)

print(
    "Attempted:",
    len(missing_images)
)

print(
    "Successfully enhanced:",
    success_count
)

print(
    "Still failed:",
    len(failed_images)
)

print(
    "\nFailure log:"
)

print(
    retry_failed_log
)

In [ ]:
import os
import cv2
from collections import Counter

# ============================================================
# PATHS
# ============================================================

original_root = "/kaggle/working/vehicle_dataset/Images"
enhanced_root = "/kaggle/working/enhanced_vehicle_dataset/Images"

valid_extensions = (
    ".jpg", ".jpeg", ".png",
    ".JPG", ".JPEG", ".PNG"
)

# ============================================================
# VALIDATION
# ============================================================

total_original = 0
total_enhanced = 0

missing_files = []
unreadable_original = []
unreadable_enhanced = []
dimension_mismatch = []

class_counts_original = Counter()
class_counts_enhanced = Counter()

# ============================================================
# CHECK EACH ORIGINAL IMAGE
# ============================================================

for class_name in sorted(os.listdir(original_root)):

    original_class_dir = os.path.join(
        original_root,
        class_name
    )

    if not os.path.isdir(original_class_dir):
        continue

    for filename in sorted(os.listdir(original_class_dir)):

        if not filename.endswith(valid_extensions):
            continue

        total_original += 1
        class_counts_original[class_name] += 1

        original_path = os.path.join(
            original_class_dir,
            filename
        )

        enhanced_path = os.path.join(
            enhanced_root,
            class_name,
            filename
        )

        # ----------------------------------------------------
        # CHECK EXISTENCE
        # ----------------------------------------------------

        if not os.path.exists(enhanced_path):

            missing_files.append(
                os.path.join(
                    class_name,
                    filename
                )
            )

            continue

        total_enhanced += 1
        class_counts_enhanced[class_name] += 1

        # ----------------------------------------------------
        # READ IMAGES
        # ----------------------------------------------------

        original_img = cv2.imread(
            original_path
        )

        enhanced_img = cv2.imread(
            enhanced_path
        )

        if original_img is None:

            unreadable_original.append(
                os.path.join(
                    class_name,
                    filename
                )
            )

            continue

        if enhanced_img is None:

            unreadable_enhanced.append(
                os.path.join(
                    class_name,
                    filename
                )
            )

            continue

        # ----------------------------------------------------
        # DIMENSION CHECK
        # ----------------------------------------------------

        original_h, original_w = original_img.shape[:2]
        enhanced_h, enhanced_w = enhanced_img.shape[:2]

        if (
            original_h != enhanced_h
            or original_w != enhanced_w
        ):

            dimension_mismatch.append(
                (
                    os.path.join(
                        class_name,
                        filename
                    ),
                    (original_w, original_h),
                    (enhanced_w, enhanced_h)
                )
            )

# ============================================================
# SUMMARY
# ============================================================

print("=" * 65)
print("FINAL ENHANCED DATASET VALIDATION")
print("=" * 65)

print("\nTotal original images:", total_original)
print("Total enhanced images:", total_enhanced)

print("\nMissing enhanced files:", len(missing_files))
print("Unreadable original files:", len(unreadable_original))
print("Unreadable enhanced files:", len(unreadable_enhanced))
print("Dimension mismatches:", len(dimension_mismatch))

print("\nOriginal class counts:")

for class_name in sorted(class_counts_original):
    print(
        f"{class_name}:",
        class_counts_original[class_name]
    )

print("\nEnhanced class counts:")

for class_name in sorted(class_counts_enhanced):
    print(
        f"{class_name}:",
        class_counts_enhanced[class_name]
    )

# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 65)

if (
    total_original == 2997
    and total_enhanced == 2997
    and len(missing_files) == 0
    and len(unreadable_original) == 0
    and len(unreadable_enhanced) == 0
    and len(dimension_mismatch) == 0
):

    print("✅ DATASET VALIDATION PASSED")
    print("All 2997 enhanced images are complete and geometry-safe.")

else:

    print("❌ DATASET VALIDATION FOUND ISSUES")

print("=" * 65)

# ============================================================
# OPTIONAL ERROR DETAILS
# ============================================================

if missing_files:
    print("\nMissing files:")
    for x in missing_files[:20]:
        print(x)

if unreadable_enhanced:
    print("\nUnreadable enhanced files:")
    for x in unreadable_enhanced[:20]:
        print(x)

if dimension_mismatch:
    print("\nFirst dimension mismatches:")
    for item in dimension_mismatch[:20]:
        print(item)

In [ ]:
import os
import shutil

export_root = "/kaggle/working/vehicle_lowlight_final"

if os.path.exists(export_root):
    shutil.rmtree(export_root)

os.makedirs(export_root, exist_ok=True)

shutil.copytree(
    "/kaggle/working/vehicle_dataset",
    os.path.join(export_root, "vehicle_dataset")
)

shutil.copytree(
    "/kaggle/working/enhanced_vehicle_dataset",
    os.path.join(export_root, "enhanced_vehicle_dataset")
)

print("✅ Final export folder created:")
print(export_root)

In [42]:
import os
import shutil

source_folder = "/kaggle/working/vehicle_lowlight_final"
zip_base = "/kaggle/working/vehicle_lowlight_final"

print("Source exists:", os.path.exists(source_folder))

zip_path = shutil.make_archive(
    zip_base,
    "zip",
    root_dir="/kaggle/working",
    base_dir="vehicle_lowlight_final"
)

print("\n✅ ZIP created successfully")
print("ZIP path:", zip_path)

size_gb = os.path.getsize(zip_path) / (1024 ** 3)

print(f"ZIP size: {size_gb:.2f} GB")

Source exists: True

✅ ZIP created successfully
ZIP path: /kaggle/working/vehicle_lowlight_final.zip
ZIP size: 1.51 GB


In [48]:
import os

print("Available input datasets:\n")

for item in os.listdir("/kaggle/input"):
    print(item)

Available input datasets:

datasets


In [49]:
import os

base = "/kaggle/input/low-light-final/vehicle_lowlight_final"

original_images = os.path.join(
    base,
    "vehicle_dataset",
    "Images"
)

annotations = os.path.join(
    base,
    "vehicle_dataset",
    "Annotations"
)

enhanced_images = os.path.join(
    base,
    "enhanced_vehicle_dataset",
    "Images"
)

print("Base exists:", os.path.exists(base))
print("Original images exists:", os.path.exists(original_images))
print("Annotations exists:", os.path.exists(annotations))
print("Enhanced images exists:", os.path.exists(enhanced_images))

Base exists: False
Original images exists: False
Annotations exists: False
Enhanced images exists: False


In [50]:
import os

base = "/kaggle/input/datasets/smnahian/low-light-final/vehicle_lowlight_final"

original_images = os.path.join(
    base,
    "vehicle_dataset",
    "Images"
)

annotations = os.path.join(
    base,
    "vehicle_dataset",
    "Annotations"
)

enhanced_images = os.path.join(
    base,
    "enhanced_vehicle_dataset",
    "Images"
)

print("Base exists:", os.path.exists(base))
print("Original images exists:", os.path.exists(original_images))
print("Annotations exists:", os.path.exists(annotations))
print("Enhanced images exists:", os.path.exists(enhanced_images))

Base exists: True
Original images exists: True
Annotations exists: True
Enhanced images exists: True


In [51]:
import os
from collections import Counter

base = "/kaggle/input/datasets/smnahian/low-light-final/vehicle_lowlight_final"

original_root = os.path.join(
    base,
    "vehicle_dataset",
    "Images"
)

annotation_root = os.path.join(
    base,
    "vehicle_dataset",
    "Annotations"
)

enhanced_root = os.path.join(
    base,
    "enhanced_vehicle_dataset",
    "Images"
)

valid_ext = (
    ".jpg", ".jpeg", ".png",
    ".JPG", ".JPEG", ".PNG"
)

def count_images(root):
    counts = Counter()
    total = 0

    for class_name in sorted(os.listdir(root)):
        class_dir = os.path.join(root, class_name)

        if not os.path.isdir(class_dir):
            continue

        count = sum(
            1 for f in os.listdir(class_dir)
            if f.endswith(valid_ext)
        )

        counts[class_name] = count
        total += count

    return total, counts


def count_annotations(root):
    counts = Counter()
    total = 0

    for class_name in sorted(os.listdir(root)):
        class_dir = os.path.join(root, class_name)

        if not os.path.isdir(class_dir):
            continue

        count = sum(
            1 for f in os.listdir(class_dir)
            if f.lower().endswith(".txt")
        )

        counts[class_name] = count
        total += count

    return total, counts


original_total, original_counts = count_images(original_root)
enhanced_total, enhanced_counts = count_images(enhanced_root)
annotation_total, annotation_counts = count_annotations(annotation_root)

print("=" * 60)
print("FINAL INPUT DATASET COUNT CHECK")
print("=" * 60)

print("\nOriginal images:", original_total)
print("Enhanced images:", enhanced_total)
print("Annotations:", annotation_total)

print("\nClass-wise comparison:\n")

classes = sorted(
    set(original_counts)
    | set(enhanced_counts)
    | set(annotation_counts)
)

for cls in classes:
    print(
        f"{cls:12s} | "
        f"Original: {original_counts[cls]:4d} | "
        f"Enhanced: {enhanced_counts[cls]:4d} | "
        f"Annotations: {annotation_counts[cls]:4d}"
    )

print("\n" + "=" * 60)

if (
    original_total == 2997
    and enhanced_total == 2997
    and annotation_total == 2997
    and original_counts == enhanced_counts == annotation_counts
):
    print("✅ FINAL INPUT DATASET VERIFIED")
else:
    print("❌ COUNT OR CLASS MISMATCH FOUND")

print("=" * 60)

FINAL INPUT DATASET COUNT CHECK

Original images: 2997
Enhanced images: 2997
Annotations: 2997

Class-wise comparison:

Bicycle      | Original:  651 | Enhanced:  651 | Annotations:  651
Boat         | Original:  679 | Enhanced:  679 | Annotations:  679
Bus          | Original:  527 | Enhanced:  527 | Annotations:  527
Car          | Original:  638 | Enhanced:  638 | Annotations:  638
Motorbike    | Original:  502 | Enhanced:  502 | Annotations:  502

✅ FINAL INPUT DATASET VERIFIED


In [52]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

base = "/kaggle/input/datasets/smnahian/low-light-final/vehicle_lowlight_final"

original_root = os.path.join(
    base,
    "vehicle_dataset",
    "Images"
)

classes = ["Bicycle", "Boat", "Bus", "Car", "Motorbike"]

rows = []

for class_name in classes:
    class_dir = os.path.join(original_root, class_name)

    for filename in sorted(os.listdir(class_dir)):
        if filename.lower().endswith((".jpg", ".jpeg", ".png")):
            rows.append({
                "class": class_name,
                "filename": filename,
                "relative_path": os.path.join(class_name, filename)
            })

df = pd.DataFrame(rows)

print("Total clean images:", len(df))

# ============================================================
# 70% TRAIN, 30% TEMP
# ============================================================

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["class"]
)

# ============================================================
# TEMP -> 15% VAL + 15% TEST
# ============================================================

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["class"]
)

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

split_df = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

# Save fixed split
split_path = "/kaggle/working/vehicle_split_70_15_15.csv"

split_df.to_csv(
    split_path,
    index=False
)

print("\nSplit counts:")
print(split_df["split"].value_counts())

print("\nClass x Split:")
print(
    pd.crosstab(
        split_df["class"],
        split_df["split"]
    )
)

print("\nSaved to:")
print(split_path)

Total clean images: 2997

Split counts:
split
train    2097
val       450
test      450
Name: count, dtype: int64

Class x Split:
split      test  train  val
class                      
Bicycle      98    456   97
Boat        102    475  102
Bus          79    369   79
Car          96    446   96
Motorbike    75    351   76

Saved to:
/kaggle/working/vehicle_split_70_15_15.csv


In [53]:
import os
import pandas as pd

# ============================================================
# PATHS
# ============================================================

base = "/kaggle/input/datasets/smnahian/low-light-final/vehicle_lowlight_final"

original_root = os.path.join(
    base,
    "vehicle_dataset",
    "Images"
)

enhanced_root = os.path.join(
    base,
    "enhanced_vehicle_dataset",
    "Images"
)

annotation_root = os.path.join(
    base,
    "vehicle_dataset",
    "Annotations"
)

split_path = "/kaggle/working/vehicle_split_70_15_15.csv"

# ============================================================
# LOAD FIXED SPLIT
# ============================================================

split_df = pd.read_csv(split_path)

missing_original = []
missing_enhanced = []
missing_annotation = []

# ============================================================
# CHECK EVERY SPLIT IMAGE
# ============================================================

for _, row in split_df.iterrows():

    class_name = row["class"]
    filename = row["filename"]

    original_path = os.path.join(
        original_root,
        class_name,
        filename
    )

    enhanced_path = os.path.join(
        enhanced_root,
        class_name,
        filename
    )

    # ExDark annotation naming:
    # image.jpg -> image.jpg.txt
    annotation_path = os.path.join(
        annotation_root,
        class_name,
        filename + ".txt"
    )

    if not os.path.exists(original_path):
        missing_original.append(
            f"{class_name}/{filename}"
        )

    if not os.path.exists(enhanced_path):
        missing_enhanced.append(
            f"{class_name}/{filename}"
        )

    if not os.path.exists(annotation_path):
        missing_annotation.append(
            f"{class_name}/{filename}.txt"
        )

# ============================================================
# SPLIT OVERLAP CHECK
# ============================================================

train_set = set(
    split_df.loc[
        split_df["split"] == "train",
        "relative_path"
    ]
)

val_set = set(
    split_df.loc[
        split_df["split"] == "val",
        "relative_path"
    ]
)

test_set = set(
    split_df.loc[
        split_df["split"] == "test",
        "relative_path"
    ]
)

train_val_overlap = train_set & val_set
train_test_overlap = train_set & test_set
val_test_overlap = val_set & test_set

# ============================================================
# RESULTS
# ============================================================

print("=" * 65)
print("SPLIT + DATASET CONSISTENCY CHECK")
print("=" * 65)

print("\nSplit rows:", len(split_df))

print("\nTrain:", len(train_set))
print("Validation:", len(val_set))
print("Test:", len(test_set))

print("\nMissing original images:", len(missing_original))
print("Missing enhanced images:", len(missing_enhanced))
print("Missing annotations:", len(missing_annotation))

print("\nTrain-Val overlap:", len(train_val_overlap))
print("Train-Test overlap:", len(train_test_overlap))
print("Val-Test overlap:", len(val_test_overlap))

print("\n" + "=" * 65)

if (
    len(split_df) == 2997
    and len(missing_original) == 0
    and len(missing_enhanced) == 0
    and len(missing_annotation) == 0
    and len(train_val_overlap) == 0
    and len(train_test_overlap) == 0
    and len(val_test_overlap) == 0
):
    print("✅ SPLIT AND DATASET CONSISTENCY VERIFIED")
    print("Original + Enhanced + Annotation pairing is complete.")
    print("No train/val/test leakage detected.")
else:
    print("❌ CONSISTENCY ISSUE FOUND")

print("=" * 65)

if missing_original:
    print("\nFirst missing originals:")
    print(missing_original[:10])

if missing_enhanced:
    print("\nFirst missing enhanced:")
    print(missing_enhanced[:10])

if missing_annotation:
    print("\nFirst missing annotations:")
    print(missing_annotation[:10])

SPLIT + DATASET CONSISTENCY CHECK

Split rows: 2997

Train: 2097
Validation: 450
Test: 450

Missing original images: 0
Missing enhanced images: 0
Missing annotations: 0

Train-Val overlap: 0
Train-Test overlap: 0
Val-Test overlap: 0

✅ SPLIT AND DATASET CONSISTENCY VERIFIED
Original + Enhanced + Annotation pairing is complete.
No train/val/test leakage detected.


In [54]:
import os
import cv2

# ============================================================
# PATHS
# ============================================================

base = "/kaggle/input/datasets/smnahian/low-light-final/vehicle_lowlight_final"

image_root = os.path.join(base, "vehicle_dataset", "Images")
annotation_root = os.path.join(base, "vehicle_dataset", "Annotations")

# Fixed class mapping
class_map = {
    "Bicycle": 0,
    "Boat": 1,
    "Bus": 2,
    "Car": 3,
    "Motorbike": 4
}

# ============================================================
# TAKE ONE SAMPLE FROM OUR LOCKED SPLIT
# ============================================================

sample = split_df.iloc[0]

class_name = sample["class"]
filename = sample["filename"]

image_path = os.path.join(
    image_root,
    class_name,
    filename
)

annotation_path = os.path.join(
    annotation_root,
    class_name,
    filename + ".txt"
)

# ============================================================
# READ IMAGE
# ============================================================

img = cv2.imread(image_path)

if img is None:
    raise ValueError("Could not read sample image.")

img_h, img_w = img.shape[:2]

print("=" * 65)
print("YOLO ANNOTATION CONVERSION TEST")
print("=" * 65)

print("\nImage:")
print(f"{class_name}/{filename}")

print(f"\nImage size: {img_w} x {img_h}")

# ============================================================
# READ + CONVERT ANNOTATION
# ============================================================

converted_labels = []

with open(annotation_path, "r") as f:
    lines = f.readlines()

for line in lines:

    line = line.strip()

    # Skip header / empty lines
    if not line or line.startswith("%"):
        continue

    parts = line.split()

    object_class = parts[0]

    # Ignore anything outside our 5 vehicle classes
    if object_class not in class_map:
        continue

    x_left = float(parts[1])
    y_top = float(parts[2])
    box_w = float(parts[3])
    box_h = float(parts[4])

    # --------------------------------------------------------
    # bbGt -> YOLO
    # --------------------------------------------------------

    x_center = (x_left + box_w / 2) / img_w
    y_center = (y_top + box_h / 2) / img_h

    norm_w = box_w / img_w
    norm_h = box_h / img_h

    class_id = class_map[object_class]

    converted_labels.append(
        [
            class_id,
            x_center,
            y_center,
            norm_w,
            norm_h
        ]
    )

    print("\nOriginal bbGt:")
    print(
        f"{object_class} "
        f"x={x_left}, y={y_top}, "
        f"w={box_w}, h={box_h}"
    )

    print("Converted YOLO:")
    print(
        f"{class_id} "
        f"{x_center:.6f} "
        f"{y_center:.6f} "
        f"{norm_w:.6f} "
        f"{norm_h:.6f}"
    )

# ============================================================
# SANITY CHECK
# ============================================================

valid = True

for label in converted_labels:
    _, xc, yc, w, h = label

    if not (
        0 <= xc <= 1 and
        0 <= yc <= 1 and
        0 < w <= 1 and
        0 < h <= 1
    ):
        valid = False

print("\n" + "=" * 65)

print("Vehicle objects found:", len(converted_labels))

if valid and len(converted_labels) > 0:
    print("✅ SAMPLE YOLO CONVERSION PASSED")
    print("All normalized coordinates are within valid range.")
else:
    print("❌ SAMPLE YOLO CONVERSION FAILED")

print("=" * 65)

YOLO ANNOTATION CONVERSION TEST

Image:
Bus/2015_02134.jpg

Image size: 640 x 640

Original bbGt:
Bus x=91.0, y=399.0, w=546.0, h=185.0
Converted YOLO:
2 0.568750 0.767969 0.853125 0.289062

Vehicle objects found: 1
✅ SAMPLE YOLO CONVERSION PASSED
All normalized coordinates are within valid range.


In [55]:
import os
import shutil
import cv2
import pandas as pd

# ============================================================
# PATHS
# ============================================================

base = "/kaggle/input/datasets/smnahian/low-light-final/vehicle_lowlight_final"

original_root = os.path.join(
    base, "vehicle_dataset", "Images"
)

enhanced_root = os.path.join(
    base, "enhanced_vehicle_dataset", "Images"
)

annotation_root = os.path.join(
    base, "vehicle_dataset", "Annotations"
)

split_path = "/kaggle/working/vehicle_split_70_15_15.csv"

output_original = "/kaggle/working/yolo_original"
output_enhanced = "/kaggle/working/yolo_enhanced"

# ============================================================
# CLASS MAPPING
# ============================================================

class_map = {
    "Bicycle": 0,
    "Boat": 1,
    "Bus": 2,
    "Car": 3,
    "Motorbike": 4
}

# ============================================================
# LOAD LOCKED SPLIT
# ============================================================

split_df = pd.read_csv(split_path)

print("Loaded split rows:", len(split_df))

# ============================================================
# CREATE DIRECTORY STRUCTURE
# ============================================================

for dataset_root in [output_original, output_enhanced]:
    for split in ["train", "val", "test"]:
        os.makedirs(
            os.path.join(dataset_root, "images", split),
            exist_ok=True
        )
        os.makedirs(
            os.path.join(dataset_root, "labels", split),
            exist_ok=True
        )

# ============================================================
# CONVERSION FUNCTION
# ============================================================

def convert_annotation(annotation_path, img_w, img_h):

    yolo_lines = []

    with open(annotation_path, "r") as f:
        lines = f.readlines()

    for line in lines:

        line = line.strip()

        if not line or line.startswith("%"):
            continue

        parts = line.split()

        object_class = parts[0]

        if object_class not in class_map:
            continue

        x_left = float(parts[1])
        y_top = float(parts[2])
        box_w = float(parts[3])
        box_h = float(parts[4])

        # bbGt -> YOLO
        x_center = (x_left + box_w / 2) / img_w
        y_center = (y_top + box_h / 2) / img_h

        norm_w = box_w / img_w
        norm_h = box_h / img_h

        class_id = class_map[object_class]

        # Safety check
        if not (
            0 <= x_center <= 1 and
            0 <= y_center <= 1 and
            0 < norm_w <= 1 and
            0 < norm_h <= 1
        ):
            raise ValueError(
                f"Invalid YOLO box in {annotation_path}: "
                f"{object_class} "
                f"{x_center}, {y_center}, {norm_w}, {norm_h}"
            )

        yolo_lines.append(
            f"{class_id} "
            f"{x_center:.6f} "
            f"{y_center:.6f} "
            f"{norm_w:.6f} "
            f"{norm_h:.6f}"
        )

    return yolo_lines


# ============================================================
# BUILD DATASETS
# ============================================================

processed = 0
total_boxes = 0

for _, row in split_df.iterrows():

    class_name = row["class"]
    filename = row["filename"]
    split = row["split"]

    original_path = os.path.join(
        original_root,
        class_name,
        filename
    )

    enhanced_path = os.path.join(
        enhanced_root,
        class_name,
        filename
    )

    annotation_path = os.path.join(
        annotation_root,
        class_name,
        filename + ".txt"
    )

    # Read original only to get dimensions
    img = cv2.imread(original_path)

    if img is None:
        raise ValueError(
            f"Could not read image: {original_path}"
        )

    img_h, img_w = img.shape[:2]

    # Convert annotation
    yolo_lines = convert_annotation(
        annotation_path,
        img_w,
        img_h
    )

    if len(yolo_lines) == 0:
        raise ValueError(
            f"No valid vehicle boxes: {annotation_path}"
        )

    total_boxes += len(yolo_lines)

    # --------------------------------------------------------
    # IMPORTANT:
    # Prevent duplicate filenames from different class folders
    # --------------------------------------------------------

    safe_name = f"{class_name}_{filename}"

    label_name = os.path.splitext(safe_name)[0] + ".txt"

    # --------------------------------------------------------
    # ORIGINAL
    # --------------------------------------------------------

    shutil.copy2(
        original_path,
        os.path.join(
            output_original,
            "images",
            split,
            safe_name
        )
    )

    with open(
        os.path.join(
            output_original,
            "labels",
            split,
            label_name
        ),
        "w"
    ) as f:
        f.write("\n".join(yolo_lines))

    # --------------------------------------------------------
    # ENHANCED
    # --------------------------------------------------------

    shutil.copy2(
        enhanced_path,
        os.path.join(
            output_enhanced,
            "images",
            split,
            safe_name
        )
    )

    # SAME LABELS because geometry is unchanged
    with open(
        os.path.join(
            output_enhanced,
            "labels",
            split,
            label_name
        ),
        "w"
    ) as f:
        f.write("\n".join(yolo_lines))

    processed += 1

# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 65)
print("FULL YOLO DATASET CREATION COMPLETE")
print("=" * 65)

print("\nImages processed:", processed)
print("Vehicle bounding boxes:", total_boxes)

for split in ["train", "val", "test"]:

    original_images = len(
        os.listdir(
            os.path.join(
                output_original,
                "images",
                split
            )
        )
    )

    original_labels = len(
        os.listdir(
            os.path.join(
                output_original,
                "labels",
                split
            )
        )
    )

    enhanced_images = len(
        os.listdir(
            os.path.join(
                output_enhanced,
                "images",
                split
            )
        )
    )

    enhanced_labels = len(
        os.listdir(
            os.path.join(
                output_enhanced,
                "labels",
                split
            )
        )
    )

    print(
        f"\n{split.upper()}:"
        f"\n  Original images : {original_images}"
        f"\n  Original labels : {original_labels}"
        f"\n  Enhanced images : {enhanced_images}"
        f"\n  Enhanced labels : {enhanced_labels}"
    )

print("\nExpected:")
print("Train = 2097")
print("Val   = 450")
print("Test  = 450")

print("\nOriginal dataset:", output_original)
print("Enhanced dataset:", output_enhanced)

print("=" * 65)

Loaded split rows: 2997

FULL YOLO DATASET CREATION COMPLETE

Images processed: 2997
Vehicle bounding boxes: 6680

TRAIN:
  Original images : 2097
  Original labels : 2097
  Enhanced images : 2097
  Enhanced labels : 2097

VAL:
  Original images : 450
  Original labels : 450
  Enhanced images : 450
  Enhanced labels : 450

TEST:
  Original images : 450
  Original labels : 450
  Enhanced images : 450
  Enhanced labels : 450

Expected:
Train = 2097
Val   = 450
Test  = 450

Original dataset: /kaggle/working/yolo_original
Enhanced dataset: /kaggle/working/yolo_enhanced


In [56]:
import os
from collections import Counter

# ============================================================
# PATHS
# ============================================================

datasets = {
    "Original": "/kaggle/working/yolo_original",
    "Enhanced": "/kaggle/working/yolo_enhanced"
}

class_names = {
    0: "Bicycle",
    1: "Boat",
    2: "Bus",
    3: "Car",
    4: "Motorbike"
}

# ============================================================
# AUDIT FUNCTION
# ============================================================

def audit_yolo_dataset(root):

    total_images = 0
    total_labels = 0
    total_boxes = 0

    class_counts = Counter()

    invalid_lines = []
    empty_labels = []
    missing_labels = []
    missing_images = []

    split_summary = {}

    for split in ["train", "val", "test"]:

        image_dir = os.path.join(root, "images", split)
        label_dir = os.path.join(root, "labels", split)

        images = sorted([
            f for f in os.listdir(image_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])

        labels = sorted([
            f for f in os.listdir(label_dir)
            if f.lower().endswith(".txt")
        ])

        total_images += len(images)
        total_labels += len(labels)

        split_boxes = 0

        # -------------------------------
        # Check images -> labels
        # -------------------------------

        for image_name in images:

            stem = os.path.splitext(image_name)[0]
            label_name = stem + ".txt"

            label_path = os.path.join(
                label_dir,
                label_name
            )

            if not os.path.exists(label_path):
                missing_labels.append(
                    f"{split}/{image_name}"
                )
                continue

            with open(label_path, "r") as f:
                lines = [
                    x.strip()
                    for x in f.readlines()
                    if x.strip()
                ]

            if len(lines) == 0:
                empty_labels.append(
                    f"{split}/{label_name}"
                )

            for line_no, line in enumerate(lines, start=1):

                parts = line.split()

                if len(parts) != 5:
                    invalid_lines.append(
                        (split, label_name, line_no, line)
                    )
                    continue

                try:
                    class_id = int(parts[0])

                    xc = float(parts[1])
                    yc = float(parts[2])
                    w = float(parts[3])
                    h = float(parts[4])

                except ValueError:

                    invalid_lines.append(
                        (split, label_name, line_no, line)
                    )
                    continue

                if (
                    class_id not in class_names
                    or not (0 <= xc <= 1)
                    or not (0 <= yc <= 1)
                    or not (0 < w <= 1)
                    or not (0 < h <= 1)
                ):
                    invalid_lines.append(
                        (split, label_name, line_no, line)
                    )
                    continue

                class_counts[class_id] += 1

                split_boxes += 1
                total_boxes += 1

        # -------------------------------
        # Check labels -> images
        # -------------------------------

        image_stems = {
            os.path.splitext(x)[0]
            for x in images
        }

        for label_name in labels:

            stem = os.path.splitext(label_name)[0]

            if stem not in image_stems:
                missing_images.append(
                    f"{split}/{label_name}"
                )

        split_summary[split] = {
            "images": len(images),
            "labels": len(labels),
            "boxes": split_boxes
        }

    return {
        "total_images": total_images,
        "total_labels": total_labels,
        "total_boxes": total_boxes,
        "class_counts": class_counts,
        "invalid_lines": invalid_lines,
        "empty_labels": empty_labels,
        "missing_labels": missing_labels,
        "missing_images": missing_images,
        "split_summary": split_summary
    }


# ============================================================
# RUN AUDIT
# ============================================================

results = {}

for name, root in datasets.items():

    results[name] = audit_yolo_dataset(root)

    r = results[name]

    print("\n" + "=" * 70)
    print(f"{name.upper()} YOLO DATASET AUDIT")
    print("=" * 70)

    for split in ["train", "val", "test"]:

        s = r["split_summary"][split]

        print(
            f"{split.upper():5} | "
            f"Images: {s['images']:4} | "
            f"Labels: {s['labels']:4} | "
            f"Boxes: {s['boxes']:4}"
        )

    print("\nTotal images :", r["total_images"])
    print("Total labels :", r["total_labels"])
    print("Total boxes  :", r["total_boxes"])

    print("\nBoxes by class:")

    for class_id, class_name in class_names.items():

        print(
            f"{class_id} {class_name:10}: "
            f"{r['class_counts'][class_id]}"
        )

    print("\nMissing labels :", len(r["missing_labels"]))
    print("Missing images :", len(r["missing_images"]))
    print("Empty labels   :", len(r["empty_labels"]))
    print("Invalid lines  :", len(r["invalid_lines"]))


# ============================================================
# ORIGINAL vs ENHANCED LABEL EQUALITY
# ============================================================

label_mismatches = []

for split in ["train", "val", "test"]:

    orig_dir = os.path.join(
        datasets["Original"],
        "labels",
        split
    )

    enh_dir = os.path.join(
        datasets["Enhanced"],
        "labels",
        split
    )

    for filename in os.listdir(orig_dir):

        orig_path = os.path.join(orig_dir, filename)
        enh_path = os.path.join(enh_dir, filename)

        with open(orig_path, "r") as f:
            orig_text = f.read()

        with open(enh_path, "r") as f:
            enh_text = f.read()

        if orig_text != enh_text:
            label_mismatches.append(
                f"{split}/{filename}"
            )

print("\n" + "=" * 70)
print("ORIGINAL vs ENHANCED LABEL CHECK")
print("=" * 70)

print("Label mismatches:", len(label_mismatches))

if (
    results["Original"]["total_images"] == 2997
    and results["Enhanced"]["total_images"] == 2997
    and len(results["Original"]["invalid_lines"]) == 0
    and len(results["Enhanced"]["invalid_lines"]) == 0
    and len(results["Original"]["missing_labels"]) == 0
    and len(results["Enhanced"]["missing_labels"]) == 0
    and len(label_mismatches) == 0
):
    print("✅ YOLO DATASET INTEGRITY PASSED")
else:
    print("❌ YOLO DATASET INTEGRITY ISSUE FOUND")

print("=" * 70)


ORIGINAL YOLO DATASET AUDIT
TRAIN | Images: 2097 | Labels: 2097 | Boxes: 4712
VAL   | Images:  450 | Labels:  450 | Boxes: 1002
TEST  | Images:  450 | Labels:  450 | Boxes:  966

Total images : 2997
Total labels : 2997
Total boxes  : 6680

Boxes by class:
0 Bicycle   : 1077
1 Boat      : 1377
2 Bus       : 689
3 Car       : 2501
4 Motorbike : 1036

Missing labels : 0
Missing images : 0
Empty labels   : 0
Invalid lines  : 0

ENHANCED YOLO DATASET AUDIT
TRAIN | Images: 2097 | Labels: 2097 | Boxes: 4712
VAL   | Images:  450 | Labels:  450 | Boxes: 1002
TEST  | Images:  450 | Labels:  450 | Boxes:  966

Total images : 2997
Total labels : 2997
Total boxes  : 6680

Boxes by class:
0 Bicycle   : 1077
1 Boat      : 1377
2 Bus       : 689
3 Car       : 2501
4 Motorbike : 1036

Missing labels : 0
Missing images : 0
Empty labels   : 0
Invalid lines  : 0

ORIGINAL vs ENHANCED LABEL CHECK
Label mismatches: 0
✅ YOLO DATASET INTEGRITY PASSED


In [57]:
import os
import yaml

# ============================================================
# DATASET PATHS
# ============================================================

original_root = "/kaggle/working/yolo_original"
enhanced_root = "/kaggle/working/yolo_enhanced"

# ============================================================
# CLASS NAMES
# ============================================================

class_names = {
    0: "Bicycle",
    1: "Boat",
    2: "Bus",
    3: "Car",
    4: "Motorbike"
}

# ============================================================
# ORIGINAL YAML
# ============================================================

original_yaml = {
    "path": original_root,
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": class_names
}

original_yaml_path = "/kaggle/working/original_vehicle.yaml"

with open(original_yaml_path, "w") as f:
    yaml.dump(
        original_yaml,
        f,
        sort_keys=False
    )

# ============================================================
# ENHANCED YAML
# ============================================================

enhanced_yaml = {
    "path": enhanced_root,
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": class_names
}

enhanced_yaml_path = "/kaggle/working/enhanced_vehicle.yaml"

with open(enhanced_yaml_path, "w") as f:
    yaml.dump(
        enhanced_yaml,
        f,
        sort_keys=False
    )

# ============================================================
# VERIFY
# ============================================================

print("=" * 65)
print("YOLO DATASET YAML CREATION")
print("=" * 65)

print("\nORIGINAL YAML:\n")
with open(original_yaml_path, "r") as f:
    print(f.read())

print("ENHANCED YAML:\n")
with open(enhanced_yaml_path, "r") as f:
    print(f.read())

print("Original YAML:", original_yaml_path)
print("Enhanced YAML:", enhanced_yaml_path)

print("\n✅ YOLO YAML FILES CREATED")
print("=" * 65)

YOLO DATASET YAML CREATION

ORIGINAL YAML:

path: /kaggle/working/yolo_original
train: images/train
val: images/val
test: images/test
names:
  0: Bicycle
  1: Boat
  2: Bus
  3: Car
  4: Motorbike

ENHANCED YAML:

path: /kaggle/working/yolo_enhanced
train: images/train
val: images/val
test: images/test
names:
  0: Bicycle
  1: Boat
  2: Bus
  3: Car
  4: Motorbike

Original YAML: /kaggle/working/original_vehicle.yaml
Enhanced YAML: /kaggle/working/enhanced_vehicle.yaml

✅ YOLO YAML FILES CREATED


In [58]:
# ============================================================
# YOLOv8 - ORIGINAL DATASET
# 5-EPOCH PIPELINE SANITY TRIAL
# ============================================================

!pip install -q ultralytics

import torch
from ultralytics import YOLO
import ultralytics

print("=" * 70)
print("YOLOv8 ORIGINAL DATASET - 5 EPOCH TRIAL")
print("=" * 70)

print("Ultralytics version:", ultralytics.__version__)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    device = 0
else:
    print("WARNING: GPU not available")
    device = "cpu"

# ------------------------------------------------------------
# Load pretrained YOLOv8 small
# ------------------------------------------------------------

model = YOLO("yolov8s.pt")

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

results = model.train(
    data="/kaggle/working/original_vehicle.yaml",

    epochs=5,

    imgsz=640,

    batch=8,

    device=device,

    workers=2,

    seed=42,

    deterministic=True,

    pretrained=True,

    project="/kaggle/working/vehicle_detection_trials",

    name="yolov8s_original_5epoch",

    exist_ok=True,

    verbose=True
)

print("\n" + "=" * 70)
print("✅ YOLOv8 ORIGINAL 5-EPOCH TRIAL COMPLETE")
print("=" * 70)

print(
    "Results saved at:",
    "/kaggle/working/vehicle_detection_trials/"
    "yolov8s_original_5epoch"
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 3.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
YOLOv8 ORIGINAL DATASET - 5 EPOCH TRIAL
Ultralytics version: 8.4.132
PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
Ultralytics 8.4.132 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_rem

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


        2/5       2.8G      1.456      1.639      1.468          6        640: 100% ━━━━━━━━━━━━ 263/263 6.8it/s 38.5s0.3ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 8.1it/s 3.6s0.1s
                   all        450       1002       0.42       0.53      0.457      0.256

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
        3/5      2.82G      1.451      1.633      1.466          6        640: 100% ━━━━━━━━━━━━ 263/263 6.9it/s 38.1s0.3ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 8.0it/s 3.6s0.1s
                   all        450       1002      0.684      0.578      0.652      0.396

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
        4/5      2.82G       1.36      1.434      1.408          6        640: 100% ━━━━━━━━━━━━ 263/263 6.9it/s 37.9s0.3ss
                 Class     Ima

In [59]:
# ============================================================
# YOLOv8s - ENHANCED DATASET
# SAME 5-EPOCH SANITY TRIAL
# ============================================================

from ultralytics import YOLO
import torch

print("=" * 70)
print("YOLOv8 ENHANCED DATASET - 5 EPOCH TRIAL")
print("=" * 70)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    device = 0
else:
    device = "cpu"

# IMPORTANT:
# Start again from the SAME pretrained YOLOv8s weights.
# Do NOT continue from the original-trained model.
model = YOLO("yolov8s.pt")

results = model.train(
    data="/kaggle/working/enhanced_vehicle.yaml",

    epochs=5,
    imgsz=640,
    batch=8,
    device=device,
    workers=2,

    seed=42,
    deterministic=True,
    pretrained=True,

    project="/kaggle/working/vehicle_detection_trials",
    name="yolov8s_enhanced_5epoch",
    exist_ok=True,

    verbose=True
)

print("\n" + "=" * 70)
print("✅ YOLOv8 ENHANCED 5-EPOCH TRIAL COMPLETE")
print("=" * 70)

print(
    "Results saved at:",
    "/kaggle/working/vehicle_detection_trials/"
    "yolov8s_enhanced_5epoch"
)

YOLOv8 ENHANCED DATASET - 5 EPOCH TRIAL
CUDA available: True
GPU: Tesla T4
Ultralytics 8.4.132 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/enhanced_vehicle.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, m

In [60]:
from ultralytics import YOLO
import torch

print("=" * 70)
print("YOLOv8s ORIGINAL - FINAL FINE-TUNING")
print("=" * 70)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    device = 0
else:
    device = "cpu"

# Start from pretrained COCO weights
# NOT from the 5-epoch trial
model = YOLO("yolov8s.pt")

results = model.train(
    data="/kaggle/working/original_vehicle.yaml",

    epochs=100,
    patience=15,

    imgsz=640,
    batch=8,

    device=device,
    workers=2,

    seed=42,
    deterministic=True,

    pretrained=True,

    project="/kaggle/working/vehicle_detection_final",
    name="yolov8s_original_final",
    exist_ok=True,

    plots=True,
    verbose=True
)

print("\n" + "=" * 70)
print("YOLOv8s ORIGINAL FINE-TUNING COMPLETE")
print("=" * 70)

print(
    "Best model:",
    "/kaggle/working/vehicle_detection_final/"
    "yolov8s_original_final/weights/best.pt"
)


YOLOv8s ORIGINAL - FINAL FINE-TUNING
CUDA available: True
GPU: Tesla T4
Ultralytics 8.4.132 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/original_vehicle.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, mo

In [61]:
from ultralytics import YOLO
import torch

print("=" * 70)
print("YOLOv8s ENHANCED - FINAL FINE-TUNING")
print("=" * 70)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    device = 0
else:
    device = "cpu"

# Start from SAME pretrained YOLOv8s weights
# Do NOT use original-trained best.pt here
model = YOLO("yolov8s.pt")

results = model.train(
    data="/kaggle/working/enhanced_vehicle.yaml",

    epochs=100,
    patience=15,

    imgsz=640,
    batch=8,

    device=device,
    workers=2,

    seed=42,
    deterministic=True,

    pretrained=True,

    project="/kaggle/working/vehicle_detection_final",
    name="yolov8s_enhanced_final",
    exist_ok=True,

    plots=True,
    verbose=True
)

print("\n" + "=" * 70)
print("YOLOv8s ENHANCED FINE-TUNING COMPLETE")
print("=" * 70)

print(
    "Best model:",
    "/kaggle/working/vehicle_detection_final/"
    "yolov8s_enhanced_final/weights/best.pt"
)

YOLOv8s ENHANCED - FINAL FINE-TUNING
CUDA available: True
GPU: Tesla T4
Ultralytics 8.4.132 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/enhanced_vehicle.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, mo

In [62]:
from ultralytics import YOLO
import torch

print("=" * 70)
print("YOLOv8s TEST 1/4")
print("ORIGINAL-TRAINED MODEL -> ORIGINAL TEST SET")
print("=" * 70)

model_path = (
    "/kaggle/working/vehicle_detection_final/"
    "yolov8s_original_final/weights/best.pt"
)

print("Model:", model_path)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    device = 0
else:
    device = "cpu"

# Load the frozen best model
model = YOLO(model_path)

# Evaluate ONLY on the untouched ORIGINAL test set
results = model.val(
    data="/kaggle/working/original_vehicle.yaml",
    split="test",
    imgsz=640,
    batch=8,
    device=device,
    workers=2,
    plots=True,
    project="/kaggle/working/vehicle_detection_test",
    name="yolov8s_Otrain_Otest",
    exist_ok=True,
    verbose=True
)

print("\n" + "=" * 70)
print("TEST 1/4 COMPLETE")
print("=" * 70)

print(f"Precision   : {results.box.mp:.6f}")
print(f"Recall      : {results.box.mr:.6f}")
print(f"mAP@50      : {results.box.map50:.6f}")
print(f"mAP@50:95   : {results.box.map:.6f}")

p = results.box.mp
r = results.box.mr
f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0

print(f"F1-score    : {f1:.6f}")

print("\nPer-class mAP@50:")
class_names = ["Bicycle", "Boat", "Bus", "Car", "Motorbike"]

for i, class_name in enumerate(class_names):
    print(
        f"{class_name:<10}: "
        f"mAP50={results.box.ap50[i]:.6f}, "
        f"mAP50-95={results.box.maps[i]:.6f}"
    )

print("\nResults saved to:")
print("/kaggle/working/vehicle_detection_test/yolov8s_Otrain_Otest")

YOLOv8s TEST 1/4
ORIGINAL-TRAINED MODEL -> ORIGINAL TEST SET
Model: /kaggle/working/vehicle_detection_final/yolov8s_original_final/weights/best.pt
CUDA available: True
GPU: Tesla T4
Ultralytics 8.4.132 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 73 layers, 11,127,519 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 66.2±57.5 MB/s, size: 125.9 KB)
val: Scanning /kaggle/working/yolo_original/labels/test... 450 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 450/450 659.5it/s 0.7s0.3s
val: /kaggle/working/yolo_original/images/test/Car_2015_02634.jpg: corrupt JPEG restored and saved
val: New cache created: /kaggle/working/yolo_original/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 57/57 9.5it/s 6.0s<0.1s
                   all        450        966      0.853      0.726      0.819      0.551
               Bicycle        108   

In [63]:
from ultralytics import YOLO
import torch

print("=" * 70)
print("YOLOv8s TEST 2/4")
print("ORIGINAL-TRAINED MODEL -> ENHANCED TEST SET")
print("=" * 70)

model_path = (
    "/kaggle/working/vehicle_detection_final/"
    "yolov8s_original_final/weights/best.pt"
)

print("Model:", model_path)

device = 0 if torch.cuda.is_available() else "cpu"

model = YOLO(model_path)

results = model.val(
    data="/kaggle/working/enhanced_vehicle.yaml",
    split="test",
    imgsz=640,
    batch=8,
    device=device,
    workers=2,
    plots=True,
    project="/kaggle/working/vehicle_detection_test",
    name="yolov8s_Otrain_Etest",
    exist_ok=True,
    verbose=True
)

print("\n" + "=" * 70)
print("TEST 2/4 COMPLETE")
print("=" * 70)

p = results.box.mp
r = results.box.mr
f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0

print(f"Precision   : {p:.6f}")
print(f"Recall      : {r:.6f}")
print(f"mAP@50      : {results.box.map50:.6f}")
print(f"mAP@50:95   : {results.box.map:.6f}")
print(f"F1-score    : {f1:.6f}")

print("\nPer-class mAP@50:")
class_names = ["Bicycle", "Boat", "Bus", "Car", "Motorbike"]

for i, class_name in enumerate(class_names):
    print(
        f"{class_name:<10}: "
        f"mAP50={results.box.ap50[i]:.6f}, "
        f"mAP50-95={results.box.maps[i]:.6f}"
    )

print("\nResults saved to:")
print("/kaggle/working/vehicle_detection_test/yolov8s_Otrain_Etest")

YOLOv8s TEST 2/4
ORIGINAL-TRAINED MODEL -> ENHANCED TEST SET
Model: /kaggle/working/vehicle_detection_final/yolov8s_original_final/weights/best.pt
Ultralytics 8.4.132 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 73 layers, 11,127,519 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 185.4±183.7 MB/s, size: 429.9 KB)
val: Scanning /kaggle/working/yolo_enhanced/labels/test... 450 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 450/450 742.0it/s 0.6s0.0s
val: New cache created: /kaggle/working/yolo_enhanced/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 57/57 10.0it/s 5.7s0.2s
                   all        450        966       0.84      0.669      0.759      0.505
               Bicycle        108        172      0.841       0.68      0.753      0.498
                  Boat        102        209      0.765      0.589      0.652  

In [64]:
from ultralytics import YOLO
import torch

print("=" * 70)
print("YOLOv8s TEST 3/4")
print("ENHANCED-TRAINED MODEL -> ORIGINAL TEST SET")
print("=" * 70)

model_path = (
    "/kaggle/working/vehicle_detection_final/"
    "yolov8s_enhanced_final/weights/best.pt"
)

print("Model:", model_path)

device = 0 if torch.cuda.is_available() else "cpu"

model = YOLO(model_path)

results = model.val(
    data="/kaggle/working/original_vehicle.yaml",
    split="test",
    imgsz=640,
    batch=8,
    device=device,
    workers=2,
    plots=True,
    project="/kaggle/working/vehicle_detection_test",
    name="yolov8s_Etrain_Otest",
    exist_ok=True,
    verbose=True
)

print("\n" + "=" * 70)
print("TEST 3/4 COMPLETE")
print("=" * 70)

p = results.box.mp
r = results.box.mr
f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0

print(f"Precision   : {p:.6f}")
print(f"Recall      : {r:.6f}")
print(f"mAP@50      : {results.box.map50:.6f}")
print(f"mAP@50:95   : {results.box.map:.6f}")
print(f"F1-score    : {f1:.6f}")

print("\nPer-class mAP@50:")
class_names = ["Bicycle", "Boat", "Bus", "Car", "Motorbike"]

for i, class_name in enumerate(class_names):
    print(
        f"{class_name:<10}: "
        f"mAP50={results.box.ap50[i]:.6f}, "
        f"mAP50-95={results.box.maps[i]:.6f}"
    )

print("\nResults saved to:")
print("/kaggle/working/vehicle_detection_test/yolov8s_Etrain_Otest")

YOLOv8s TEST 3/4
ENHANCED-TRAINED MODEL -> ORIGINAL TEST SET
Model: /kaggle/working/vehicle_detection_final/yolov8s_enhanced_final/weights/best.pt
Ultralytics 8.4.132 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 73 layers, 11,127,519 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1635.1±526.7 MB/s, size: 126.3 KB)
val: Scanning /kaggle/working/yolo_original/labels/test.cache... 450 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 450/450 118.0Mit/s 0.0s
val: /kaggle/working/yolo_original/images/test/Car_2015_02634.jpg: corrupt JPEG restored and saved
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 57/57 9.9it/s 5.7s<0.2s
                   all        450        966      0.832      0.675      0.783      0.508
               Bicycle        108        172      0.838      0.686      0.795      0.519
                  Boat        102        209    

In [65]:
from ultralytics import YOLO
import torch

print("=" * 70)
print("YOLOv8s TEST 4/4")
print("ENHANCED-TRAINED MODEL -> ENHANCED TEST SET")
print("=" * 70)

model_path = (
    "/kaggle/working/vehicle_detection_final/"
    "yolov8s_enhanced_final/weights/best.pt"
)

print("Model:", model_path)

device = 0 if torch.cuda.is_available() else "cpu"

model = YOLO(model_path)

results = model.val(
    data="/kaggle/working/enhanced_vehicle.yaml",
    split="test",
    imgsz=640,
    batch=8,
    device=device,
    workers=2,
    plots=True,
    project="/kaggle/working/vehicle_detection_test",
    name="yolov8s_Etrain_Etest",
    exist_ok=True,
    verbose=True
)

print("\n" + "=" * 70)
print("TEST 4/4 COMPLETE")
print("=" * 70)

p = results.box.mp
r = results.box.mr
f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0

print(f"Precision   : {p:.6f}")
print(f"Recall      : {r:.6f}")
print(f"mAP@50      : {results.box.map50:.6f}")
print(f"mAP@50:95   : {results.box.map:.6f}")
print(f"F1-score    : {f1:.6f}")

print("\nPer-class mAP@50:")
class_names = ["Bicycle", "Boat", "Bus", "Car", "Motorbike"]

for i, class_name in enumerate(class_names):
    print(
        f"{class_name:<10}: "
        f"mAP50={results.box.ap50[i]:.6f}, "
        f"mAP50-95={results.box.maps[i]:.6f}"
    )

print("\nResults saved to:")
print("/kaggle/working/vehicle_detection_test/yolov8s_Etrain_Etest")

YOLOv8s TEST 4/4
ENHANCED-TRAINED MODEL -> ENHANCED TEST SET
Model: /kaggle/working/vehicle_detection_final/yolov8s_enhanced_final/weights/best.pt
Ultralytics 8.4.132 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 73 layers, 11,127,519 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1501.8±590.8 MB/s, size: 105.5 KB)
val: Scanning /kaggle/working/yolo_enhanced/labels/test.cache... 450 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 450/450 209.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 57/57 9.8it/s 5.8s<0.2s
                   all        450        966      0.823      0.697      0.799      0.526
               Bicycle        108        172      0.818      0.738      0.794      0.531
                  Boat        102        209      0.751      0.578      0.682      0.353
                   Bus         86         93       0.93   

In [67]:
import os
import time
from pathlib import Path

import torch
import numpy as np
from ultralytics import YOLO

print("=" * 70)
print("CONTROLLED FPS BENCHMARK - CORRECTED")
print("YOLOv8s ORIGINAL-TRAINED -> ORIGINAL TEST")
print("=" * 70)

model_path = (
    "/kaggle/working/vehicle_detection_final/"
    "yolov8s_original_final/weights/best.pt"
)

test_dir = Path("/kaggle/working/yolo_original/images/test")

device = 0 if torch.cuda.is_available() else "cpu"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# --------------------------------------------------
# FIND ALL SUPPORTED IMAGE FORMATS
# --------------------------------------------------
valid_exts = {
    ".jpg", ".jpeg", ".png", ".bmp",
    ".tif", ".tiff", ".webp"
}

image_paths = sorted([
    str(p)
    for p in test_dir.iterdir()
    if p.is_file() and p.suffix.lower() in valid_exts
])

print("Test images found:", len(image_paths))

if len(image_paths) != 450:
    print("⚠️ WARNING: Expected 450 test images, found:", len(image_paths))
else:
    print("✅ All 450 test images detected.")

# --------------------------------------------------
# LOAD MODEL
# --------------------------------------------------
model = YOLO(model_path)

# --------------------------------------------------
# WARM-UP
# --------------------------------------------------
print("\nRunning warm-up...")

for img in image_paths[:20]:
    _ = model.predict(
        source=img,
        imgsz=640,
        device=device,
        verbose=False
    )

if torch.cuda.is_available():
    torch.cuda.synchronize()

print("Warm-up complete.")

# --------------------------------------------------
# CONTROLLED BENCHMARK
# --------------------------------------------------
times = []

print("\nBenchmarking all test images...")

for img in image_paths:

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.perf_counter()

    _ = model.predict(
        source=img,
        imgsz=640,
        device=device,
        verbose=False
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    end = time.perf_counter()

    times.append(end - start)

times = np.asarray(times)

mean_latency = times.mean()
median_latency = np.median(times)

mean_fps = 1 / mean_latency
median_fps = 1 / median_latency

total_time = times.sum()
throughput_fps = len(times) / total_time

print("\n" + "=" * 70)
print("FINAL CONTROLLED FPS RESULT")
print("=" * 70)

print(f"Images tested          : {len(times)}")
print(f"Total inference time   : {total_time:.3f} s")

print(f"Mean latency           : {mean_latency * 1000:.3f} ms/image")
print(f"Median latency         : {median_latency * 1000:.3f} ms/image")

print(f"Mean-latency FPS       : {mean_fps:.2f}")
print(f"Median-latency FPS     : {median_fps:.2f}")
print(f"Overall throughput FPS : {throughput_fps:.2f}")

print(f"Min latency            : {times.min() * 1000:.3f} ms")
print(f"Max latency            : {times.max() * 1000:.3f} ms")

print("=" * 70)

CONTROLLED FPS BENCHMARK - CORRECTED
YOLOv8s ORIGINAL-TRAINED -> ORIGINAL TEST
CUDA available: True
GPU: Tesla T4
Test images found: 450
✅ All 450 test images detected.

Running warm-up...
Warm-up complete.

Benchmarking all test images...

FINAL CONTROLLED FPS RESULT
Images tested          : 450
Total inference time   : 7.555 s
Mean latency           : 16.789 ms/image
Median latency         : 13.207 ms/image
Mean-latency FPS       : 59.56
Median-latency FPS     : 75.72
Overall throughput FPS : 59.56
Min latency            : 9.935 ms
Max latency            : 102.479 ms


In [68]:
import time
from pathlib import Path

import torch
import numpy as np
from ultralytics import YOLO

print("=" * 80)
print("YOLOv8s - REMAINING CONTROLLED FPS BENCHMARKS")
print("=" * 80)

# ============================================================
# CONFIGURATION
# ============================================================

ORIGINAL_MODEL = (
    "/kaggle/working/vehicle_detection_final/"
    "yolov8s_original_final/weights/best.pt"
)

ENHANCED_MODEL = (
    "/kaggle/working/vehicle_detection_final/"
    "yolov8s_enhanced_final/weights/best.pt"
)

ORIGINAL_TEST = Path(
    "/kaggle/working/yolo_original/images/test"
)

ENHANCED_TEST = Path(
    "/kaggle/working/yolo_enhanced/images/test"
)

device = 0 if torch.cuda.is_available() else "cpu"

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Device:", device)

valid_exts = {
    ".jpg", ".jpeg", ".png", ".bmp",
    ".tif", ".tiff", ".webp"
}


# ============================================================
# BENCHMARK FUNCTION
# ============================================================

def run_fps_benchmark(model_path, test_dir, experiment_name):

    print("\n" + "=" * 80)
    print(experiment_name)
    print("=" * 80)

    # --------------------------------------------------------
    # Find ALL supported images
    # --------------------------------------------------------

    image_paths = sorted([
        str(p)
        for p in test_dir.iterdir()
        if p.is_file() and p.suffix.lower() in valid_exts
    ])

    print("Test images found:", len(image_paths))

    if len(image_paths) == 450:
        print("✅ All 450 test images detected.")
    else:
        print(
            f"⚠️ WARNING: Expected 450 test images, "
            f"found {len(image_paths)}"
        )

    # --------------------------------------------------------
    # Load model
    # --------------------------------------------------------

    print("Loading model...")

    model = YOLO(model_path)

    # --------------------------------------------------------
    # Warm-up
    # --------------------------------------------------------

    print("Running 20-image warm-up...")

    for img in image_paths[:20]:

        _ = model.predict(
            source=img,
            imgsz=640,
            device=device,
            verbose=False
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    print("Warm-up complete.")

    # --------------------------------------------------------
    # Controlled timing
    # --------------------------------------------------------

    print("Benchmarking...")

    times = []

    for img in image_paths:

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start = time.perf_counter()

        _ = model.predict(
            source=img,
            imgsz=640,
            device=device,
            verbose=False
        )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        end = time.perf_counter()

        times.append(end - start)

    times = np.asarray(times)

    # --------------------------------------------------------
    # Calculate metrics
    # --------------------------------------------------------

    total_time = times.sum()

    mean_latency = times.mean()
    median_latency = np.median(times)

    throughput_fps = len(times) / total_time

    mean_latency_fps = 1 / mean_latency
    median_latency_fps = 1 / median_latency

    min_latency = times.min()
    max_latency = times.max()

    # --------------------------------------------------------
    # Print result
    # --------------------------------------------------------

    print("\nRESULT")

    print(f"Images tested          : {len(times)}")
    print(f"Total inference time   : {total_time:.3f} s")

    print(
        f"Mean latency           : "
        f"{mean_latency * 1000:.3f} ms/image"
    )

    print(
        f"Median latency         : "
        f"{median_latency * 1000:.3f} ms/image"
    )

    print(
        f"Mean-latency FPS       : "
        f"{mean_latency_fps:.2f}"
    )

    print(
        f"Median-latency FPS     : "
        f"{median_latency_fps:.2f}"
    )

    print(
        f"Overall throughput FPS : "
        f"{throughput_fps:.2f}"
    )

    print(
        f"Min latency            : "
        f"{min_latency * 1000:.3f} ms"
    )

    print(
        f"Max latency            : "
        f"{max_latency * 1000:.3f} ms"
    )

    return {
        "Experiment": experiment_name,
        "Images": len(times),
        "Mean_ms": mean_latency * 1000,
        "Median_ms": median_latency * 1000,
        "FPS": throughput_fps
    }


# ============================================================
# TEST 2
# Original-trained -> Enhanced Test
# ============================================================

result_2 = run_fps_benchmark(
    ORIGINAL_MODEL,
    ENHANCED_TEST,
    "TEST 2: ORIGINAL-TRAINED -> ENHANCED TEST"
)


# ============================================================
# TEST 3
# Enhanced-trained -> Original Test
# ============================================================

result_3 = run_fps_benchmark(
    ENHANCED_MODEL,
    ORIGINAL_TEST,
    "TEST 3: ENHANCED-TRAINED -> ORIGINAL TEST"
)


# ============================================================
# TEST 4
# Enhanced-trained -> Enhanced Test
# ============================================================

result_4 = run_fps_benchmark(
    ENHANCED_MODEL,
    ENHANCED_TEST,
    "TEST 4: ENHANCED-TRAINED -> ENHANCED TEST"
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n\n" + "=" * 100)
print("YOLOv8s CONTROLLED FPS SUMMARY")
print("=" * 100)

print(
    f"{'Condition':<50}"
    f"{'Images':>10}"
    f"{'Mean ms':>12}"
    f"{'Median ms':>14}"
    f"{'FPS':>12}"
)

print("-" * 100)

# Already measured Test 1 result
print(
    f"{'Original-trained -> Original Test':<50}"
    f"{450:>10}"
    f"{16.789:>12.3f}"
    f"{13.207:>14.3f}"
    f"{59.56:>12.2f}"
)

for r in [result_2, result_3, result_4]:

    short_name = r["Experiment"].split(": ", 1)[-1]

    print(
        f"{short_name:<50}"
        f"{r['Images']:>10}"
        f"{r['Mean_ms']:>12.3f}"
        f"{r['Median_ms']:>14.3f}"
        f"{r['FPS']:>12.2f}"
    )

print("=" * 100)

print("\n✅ ALL YOLOv8s FPS BENCHMARKS COMPLETE")

YOLOv8s - REMAINING CONTROLLED FPS BENCHMARKS
CUDA available: True
GPU: Tesla T4
Device: 0

TEST 2: ORIGINAL-TRAINED -> ENHANCED TEST
Test images found: 450
✅ All 450 test images detected.
Loading model...
Running 20-image warm-up...
Warm-up complete.
Benchmarking...

RESULT
Images tested          : 450
Total inference time   : 7.404 s
Mean latency           : 16.453 ms/image
Median latency         : 12.737 ms/image
Mean-latency FPS       : 60.78
Median-latency FPS     : 78.51
Overall throughput FPS : 60.78
Min latency            : 9.517 ms
Max latency            : 140.929 ms

TEST 3: ENHANCED-TRAINED -> ORIGINAL TEST
Test images found: 450
✅ All 450 test images detected.
Loading model...
Running 20-image warm-up...
Warm-up complete.
Benchmarking...

RESULT
Images tested          : 450
Total inference time   : 7.462 s
Mean latency           : 16.581 ms/image
Median latency         : 12.945 ms/image
Mean-latency FPS       : 60.31
Median-latency FPS     : 77.25
Overall throughput FPS : 6

In [69]:
from ultralytics import YOLO
import torch

print("=" * 70)
print("YOLO11s ORIGINAL - FINAL FINE-TUNING")
print("=" * 70)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    device = 0
else:
    device = "cpu"

# IMPORTANT:
# Start fresh from COCO-pretrained YOLO11s
model = YOLO("yolo11s.pt")

results = model.train(
    data="/kaggle/working/original_vehicle.yaml",

    epochs=100,
    patience=15,

    imgsz=640,
    batch=8,

    device=device,
    workers=2,

    seed=42,
    deterministic=True,

    pretrained=True,

    project="/kaggle/working/vehicle_detection_final",
    name="yolo11s_original_final",
    exist_ok=True,

    plots=True,
    verbose=True
)

print("\n" + "=" * 70)
print("YOLO11s ORIGINAL FINE-TUNING COMPLETE")
print("=" * 70)

print(
    "Best model:",
    "/kaggle/working/vehicle_detection_final/"
    "yolo11s_original_final/weights/best.pt"
)

YOLO11s ORIGINAL - FINAL FINE-TUNING
CUDA available: True
GPU: Tesla T4
New https://pypi.org/project/ultralytics/8.4.133 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.132 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/original_vehicle.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_

In [ ]:
from ultralytics import YOLO
import torch

print("=" * 70)
print("YOLO11s ENHANCED - FINAL FINE-TUNING")
print("=" * 70)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    device = 0
else:
    device = "cpu"

# IMPORTANT:
# Fresh COCO-pretrained YOLO11s.
# Do NOT use the Original-trained checkpoint.
model = YOLO("yolo11s.pt")

results = model.train(
    data="/kaggle/working/enhanced_vehicle.yaml",

    epochs=100,
    patience=15,

    imgsz=640,
    batch=8,

    device=device,
    workers=2,

    seed=42,
    deterministic=True,

    pretrained=True,

    project="/kaggle/working/vehicle_detection_final",
    name="yolo11s_enhanced_final",
    exist_ok=True,

    plots=True,
    verbose=True
)

print("\n" + "=" * 70)
print("YOLO11s ENHANCED FINE-TUNING COMPLETE")
print("=" * 70)

print(
    "Best model:",
    "/kaggle/working/vehicle_detection_final/"
    "yolo11s_enhanced_final/weights/best.pt"
)

YOLO11s ENHANCED - FINAL FINE-TUNING
CUDA available: True
GPU: Tesla T4
New https://pypi.org/project/ultralytics/8.4.133 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.132 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/enhanced_vehicle.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_